In [ ]:
import gc

try:
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception:
    gc.collect()
    print('Torch not available, skip GPU cache clear.')


# 第 3 课：Embeddings 与向量检索

前两课我们已经搞懂了：
- 什么是 RAG
- 为什么需要 chunking

这一课我们进入更接近真实项目的一层：

**Embedding 与向量检索**

这是很多现代 RAG 系统的核心组成部分。


## 1. 为什么 bag-of-words 不够

前两课我们用的是最小版检索器：
- 词频
- 余弦相似度

它的优点是简单，适合教学。
但它有一个明显问题：

**它更像是在匹配字面词，而不是理解语义。**

比如：
- `car`
- `automobile`

意思很接近，但如果字面词不同，bag-of-words 往往就不太好处理。


## 2. Embedding 是什么

`Embedding` 可以理解成：

**把文本映射成一个向量，让语义相近的文本在向量空间里也更接近。**

比如一句话经过 embedding 模型后，可能会变成一个 768 维或 1536 维向量。

然后我们就可以：
- 把 query 编码成向量
- 把 chunk 编码成向量
- 用向量相似度找最相关的 chunk

这就叫向量检索。


## 3. 这一课怎么讲 Embedding

真实项目里你通常会调用：
- OpenAI embedding 模型
- Hugging Face embedding 模型
- 或本地 sentence-transformers 模型

但为了保证这节 notebook 开箱即用、不会被环境卡住，
这一课我先用一个“教学版 embedding”来演示概念。

重点是让你看懂：
- query -> embedding
- chunk -> embedding
- 相似度 -> top-k 检索

后面我们再接真实模型。


In [ ]:
import math
import random
import re
from collections import defaultdict

random.seed(42)


In [ ]:
chunks = [
    'RAG retrieves external knowledge before generation.',
    'Chunking splits documents into smaller passages for retrieval.',
    'Embeddings map text into vectors so semantic similarity can be measured.',
    'Evaluation checks retrieval quality, groundedness, and final answer correctness.',
    'Vector databases store embeddings and support similarity search at scale.'
]

for i, chunk in enumerate(chunks, start=1):
    print(f'chunk {i}: {chunk}')


## 4. 一个“教学版 embedding”

为了讲清楚原理，我们这里做一个简化 embedding：

- 给每个 token 分配一个随机小向量
- 一段文本的 embedding = 所有 token 向量的平均

这当然不是真正工业级 embedding 模型，
但它足够帮助你理解 embedding 检索的数学流程。


In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z]+", text.lower())


token_to_vec = {}
embedding_dim = 8

def get_token_vector(token):
    if token not in token_to_vec:
        rnd = random.Random(hash(token) & 0xffffffff)
        token_to_vec[token] = [rnd.uniform(-1, 1) for _ in range(embedding_dim)]
    return token_to_vec[token]


def embed_text(text):
    tokens = tokenize(text)
    if not tokens:
        return [0.0] * embedding_dim

    vecs = [get_token_vector(token) for token in tokens]
    avg = []
    for i in range(embedding_dim):
        avg.append(sum(v[i] for v in vecs) / len(vecs))
    return avg


chunk_embeddings = [embed_text(chunk) for chunk in chunks]
print('embedding 维度:', embedding_dim)
print('第一个 chunk 的 embedding:', [round(x, 3) for x in chunk_embeddings[0]])


## 5. 向量相似度怎么计算

向量检索最常见的相似度之一就是：

**余弦相似度（cosine similarity）**

它本质上是在比较两个向量方向是否接近。

如果两个 embedding 方向很像，通常就表示语义更接近。


In [ ]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


sim_1_3 = cosine_similarity(chunk_embeddings[0], chunk_embeddings[2])
sim_1_4 = cosine_similarity(chunk_embeddings[0], chunk_embeddings[3])

print('chunk1 vs chunk3:', round(sim_1_3, 3))
print('chunk1 vs chunk4:', round(sim_1_4, 3))


## 6. 一个最小向量检索器

现在我们就可以把真正的 embedding retrieval 流程串起来了：

1. query 做 embedding
2. 每个 chunk 都已经有 embedding
3. 计算 query 和每个 chunk 的相似度
4. 选 top-k

这就是最小版的向量检索器。


In [ ]:
def retrieve_with_embeddings(query, chunks, chunk_embeddings, top_k=2):
    query_embedding = embed_text(query)
    scored = []
    for chunk, emb in zip(chunks, chunk_embeddings):
        score = cosine_similarity(query_embedding, emb)
        scored.append((score, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


query = 'How do embeddings help semantic retrieval?'
results = retrieve_with_embeddings(query, chunks, chunk_embeddings, top_k=3)

print('query:', query)
print()
for score, chunk in results:
    print(f'score={score:.3f}')
    print(chunk)
    print()


## 7. 和 bag-of-words 的区别到底在哪

你现在可以把两种检索方式做一个直观对比：

- `bag-of-words`
  更依赖字面词重合

- `embedding retrieval`
  更希望相近语义在向量空间里也更接近

真实 embedding 模型会比我们这个 toy 版本强很多，
因为它不是随机 token 向量平均，而是从大规模语料里学出来的语义表示。


## 8. 向量数据库在这里扮演什么角色

如果 chunk 很少，我们可以直接：
- 遍历所有向量
- 一个个算相似度

但真实项目里可能有：
- 几十万 chunk
- 几百万 chunk
- 更多

这时候就需要向量数据库或近似最近邻索引来加速检索。

所以你可以把向量数据库理解成：

**帮你在大量 embedding 里快速找到最相近向量的系统。**


In [ ]:
query2 = 'How do we evaluate a RAG system?'
results2 = retrieve_with_embeddings(query2, chunks, chunk_embeddings, top_k=2)

print('query:', query2)
for score, chunk in results2:
    print(f'score={score:.3f}')
    print(chunk)
    print()


## 9. 这节课里的最小 Evaluation 视角

Embedding 检索的效果，通常至少会从这些角度去看：

- top-k 里有没有相关 chunk
- 相比 bag-of-words，相关 chunk 排名有没有更靠前
- query 的改写 / 同义表达时是否还稳定

所以 embedding 检索的一个核心价值就是：

**让检索对语义更敏感，而不是只对字面词敏感。**


In [ ]:
gold_chunk = 'Embeddings map text into vectors so semantic similarity can be measured.'
retrieved_chunks = {chunk for _, chunk in results}
hit = gold_chunk in retrieved_chunks

print('最小 evaluation 示例:')
print('hit@3 =', hit)


## 10. 这节课最该记住什么

如果你想抓住这节课的核心主线，就记住：

- embedding 是把文本变成语义向量
- 向量检索就是在向量空间里找最相近的 chunk
- 向量数据库只是把这件事在大规模数据上做快

所以 embedding retrieval 的核心，不是数据库本身，而是：

**先把语义表示出来，再按语义做检索。**


## 11. 下一课最自然学什么

学完这一课后，最自然的下一步就是：

**Reranking / 重排序**

因为真实 RAG 系统里，向量检索常常只是第一轮召回，后面还会再做一次更精细的排序。
